# Python Spark Structured Streaming Exercises


##1. Weblog Analysis

Consider a stream of logging *events* for the web accesses.

Each logging event contain ***json*** lines as shown below:

```json
{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}```

In [ ]:
#@title Start the Structured Source

!wget -q -O - https://github.com/smduarte/spbd-2526/raw/main/docs/labs/lab7/json_logsender.tgz | tar xfz - 2> /dev/null

!nohup python json_logsender/server.py json_logsender/web.log 8888 > /dev/null 2> /dev/null &

In [ ]:
#@title Structured Streaming Session Example

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777

rawlines = spark.readStream.format("socket") \
    .option("host", "localhost") \
    .option("port", 8888) \
    .load()

query = rawlines \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='1 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

query.awaitTermination(20)
query.stop()

In [ ]:
#@title Structured Streaming Session Example

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()


# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

rawlines = spark.readStream.format("socket") \
    .option("host", "localhost") \
    .option("port", 8888) \
    .load()

json_lines = rawlines.select(from_json(col("value"), inferred_schema).alias("data")) \
 .select("data.*")

query = json_lines \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='1 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

query.awaitTermination(20)
query.stop()

# Exercises

***Every 3 seconds***,

1. Dump the number of requests in the last 10 seconds;
2. Dump the number of requests in the last 10 seconds, only if they total more than 100;
3. Dump the number of requests in the last 10 seconds, if there is an IP address with more than 100 requests;
4. Dump the proportion of IPv4 vs IPv6 requests in the last 20 seconds.


In [ ]:
#@title Q1 (incremental solution)

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  requests = json_lines \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withWatermark("date", "0 seconds") \
      .withColumn("interval", window("date", "10 seconds").start) \
      .groupBy("interval").count() \
      .withColumnRenamed("count", "#requests") \
      .orderBy('interval', ascending = False)\
      .limit(1) \

  query = requests \
    .writeStream \
    .outputMode("complete") \
    .trigger(processingTime='3 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

  query.awaitTermination(160)
except Exception as err:
  print(err)

In [ ]:
#@title Q1 (non incremental solution, incorrect)

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("ip", StringType(), True),
    StructField("code", LongType(), True),
    StructField("cmd", StringType(), True),
    StructField("url", StringType(), True),
    StructField("time", DoubleType(), True)
])

# Create DF to accumulate incoming stream data.
accumulated_df = spark.createDataFrame([], schema)
accumulated_df.printSchema()

def processBatch( df, epoch ):
  global accumulated_df
  accumulated_df = accumulated_df.union( df )

  result = accumulated_df.withColumn("date", to_timestamp(col("timestamp"))) \
      .withColumn("interval", window("date", "10 seconds").start) \
      .groupBy("interval").count() \
      .withColumnRenamed("count", "#requests") \
      .orderBy('interval', ascending = False).limit(1) \

  result.show()
  print('epoch: {}, accumulated rows: {}'.format(epoch, accumulated_df.count()))


# Create DataFrame representing the stream of input
# lines from connection to logsender 8888
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  query = json_lines \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='3 seconds') \
    .foreachBatch( lambda df, epoch: processBatch(df, epoch) ) \
    .start()

  query.awaitTermination(160)
except Exception as err:
  print(err)

In [ ]:
#@title Q2 (incremental solution)

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  requests = json_lines \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withWatermark("date", "0 seconds") \
      .withColumn("interval", window("date", "10 seconds").start) \
      .groupBy("interval").count() \
      .filter('count > 100') \
      .withColumnRenamed("count", "#requests") \
      .orderBy('interval', ascending = False)\
      .limit(1) \

  query = requests \
    .writeStream \
    .outputMode("complete") \
    .trigger(processingTime='3 seconds') \
    .foreachBatch(lambda df, epoch: df.show(10, False)) \
    .start()

  query.awaitTermination(160)
except Exception as err:
  print(err)

## Q3

Spark Structured Streaming, currently, cannot do more than aggregation per stream. For example, cannot do a ***groupBy*** followed by another ***groupBy***.


### Alternative solutions.

1. Stream-Stream join. Basically, the result will be the intersection of two streams.

2. Stream aggregation, then second aggregation plus join, in batch mode.

3. Stream aggregation then batch, then second aggregation without join (via aggregation) in batch mode.



In [ ]:
#@title Q3 (incremental solution, stream-stream join)


# Too slow for colab. Does not generate output...

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)


# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  requests = json_lines \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withColumn("interval", window("date", "2 seconds").start) \
      .withWatermark("interval", "2 seconds") \

  total_per_interval = requests.groupBy('interval') \
                      .count() \
                      .withColumnRenamed('count', '#requests')

  per_ip_counts = requests.groupBy('interval', 'ip') \
                .count() \
                .filter('count > 100') \
                .select('interval') \
                .distinct()

  # compute the intersection of the two streams...
  results = total_per_interval.join(per_ip_counts, on='interval', how='inner') \
              .select("interval", "#requests") \
              .distinct()

  query = results \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='2 seconds') \
    .foreachBatch( processBatch ) \
    .start()

  query.awaitTermination(1300)
except Exception as err:
  print(err)

In [ ]:
#@title Q3 (incremental solution 2)

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

def processBatch( df, epoch):
  total_counts = df.groupBy('interval') \
        .agg(sum('count').alias('total_count'))

  per_ip_counts = df.where('count > 100') \
        .select('interval') \
        .distinct() \
        .orderBy('interval') \

  print('intervals+counts > 100')
  results = per_ip_counts.join(total_counts, on='interval', how='inner') \

  results.show(1000)

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  per_ip_counts = json_lines \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withColumn("interval", window("date", "10 seconds").start) \
      .withWatermark("interval", "1 seconds") \
      .groupBy('interval', 'ip') \
      .count()

  query = per_ip_counts \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='2 seconds') \
    .foreachBatch( processBatch ) \
    .start()

  query.awaitTermination(1300)
except Exception as err:
  print(err)

In [ ]:
#@title Q3 (incremental solution 3)

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

def processBatch( df, epoch):
  #avoid join between dfs.
  total_counts = df.groupBy('interval') \
        .agg(
            sum('count').alias('#requests'),
            max(when(col('count') > 100, 1).otherwise(0)).alias("count_reached")) \
        .where('count_reached > 0') \
        .drop('count_reached') \
        .orderBy('interval', ascending=False) \

  total_counts.show(1000)

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  per_ip_counts = json_lines \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withColumn("interval", window("date", "10 seconds").start) \
      .withWatermark("interval", "3 seconds") \
      .groupBy('interval', 'ip') \
      .count()

  query = per_ip_counts \
    .writeStream \
    .outputMode("append") \
    .trigger(processingTime='3 seconds') \
    .foreachBatch( processBatch ) \
    .start()

  query.awaitTermination(1300)
except Exception as err:
  print(err)

In [ ]:
#@title Q4 (incremental solution)

from pyspark.sql import *
from pyspark.sql.functions import *

spark = SparkSession \
    .builder \
    .appName("StructuredWebLogExample") \
    .getOrCreate()

# Extract a sample JSON string to infer schema
sample_json = '{"timestamp": "2024-11-13T10:50:59.936+0000", "ip": "37.139.9.11", "code": 404, "cmd": "GET", "url": "/codemove/TTCENCUFMH3C", "time": 0.026}'
inferred_schema = schema_of_json(sample_json)

def processBatch( df, epoch):
  total_counts = df.groupBy('interval') \
        .agg(
            sum('count').alias('#requests'),
            max(when(col('count') > 100, 1).otherwise(0)).alias("count_reached")) \
        .where('count_reached > 0') \
        .drop('count_reached') \
        .orderBy('interval', ascending=False) \

  total_counts.show(1000)

# Create DataFrame representing the stream of input
# lines from connection to logsender 7777
try:
  json_lines = spark.readStream.format("socket") \
      .option("host", "localhost") \
      .option("port", 8888) \
      .load()

  # Parse the JSON using the inferred schema
  json_lines = json_lines.withColumn("json_data", from_json(col("value"), inferred_schema)) \
    .select("json_data.*")  # Expand the JSON fields into columns

  json_lines.printSchema()

  ip_version_counts = json_lines \
      .select('timestamp', 'ip') \
      .withColumn("date", to_timestamp(col("timestamp"))) \
      .withColumn("interval", window("date", "10 seconds").start) \
      .withWatermark("interval", "0 seconds") \
      .groupBy('interval') \
      .agg(
          count('*').alias('total'),
          sum(when(col("ip").contains("."), 1).otherwise(0)).alias("ipv4_sum"),
          sum(when(col("ip").contains(":"), 1).otherwise(0)).alias("ipv6_sum")
      ).orderBy('interval', ascending=False) \
      .limit(1) \
      .selectExpr('interval', 'round(100*ipv4_sum/total,2) as ipv4_pc', 'round(100*ipv6_sum/total,2) as ipv6_pc')

  query = ip_version_counts \
    .writeStream \
    .outputMode("complete") \
    .trigger(processingTime='3 seconds') \
    .foreachBatch( lambda df, _ : df.show(100) ) \
    .start()

  query.awaitTermination(1300)
except Exception as err:
  print(err)